# Embedding Responses to Aromatic Ring Substitution

## Setup

In [ ]:
from setup import *

from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap, ListedColormap, to_rgb
from matplotlib.lines import Line2D

## Compute Ring Embeddings

In [ ]:
n_confs = 32
model_paths = {name: CHECKPOINTS[name] for name in ("nopre", "NT", "ANT")}

ring_smiles = ["c1ccccc1",
               "c1ncccc1",
               "c1nnccc1", "c1ncncc1", "c1nccnc1",
               "c1nnncc1", "c1nncnc1", "c1ncncn1",
               "c1nnnnc1", "c1nnncn1", "c1nncnn1",
               "c1nnnnn1",
               "n1nnnnn1"]
ring_counts = [0, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 6]
ring_isomers = [0, 0, 0, 1, 2, 0, 1, 2, 0, 1, 2, 0, 0]  # which positional isomer within a count
heteroatoms = {"N": "n", "P": "p"}
substituted_smiles = ring_smiles[:1] + [s.replace("n", sym) for sym in heteroatoms.values() for s in ring_smiles[1:]]
ring_labels = ["benzene"] + [f"{X}{n}-{i}" for X in heteroatoms for n, i in zip(ring_counts[1:], ring_isomers[1:])]
batches, mol_idx = batch_smiles_conformers(substituted_smiles, n_confs)
pca_coordinates, parent_similarities = {}, {}
for name, path in model_paths.items():
    embeddings = encode_batches(EncoderModel.load_model(path, device=DEVICE).eval(), batches).numpy()
    pca_coordinates[name] = compute_pca(embeddings)
    means = average_embeddings(embeddings, len(substituted_smiles))
    parent_similarities[name] = means @ means[0]

## Figure S8: Nitrogen and Phosphorus Ring Substitution

In [ ]:
def class_gradient(anchor, n_steps, tint=0.6, shade=0.35):
    """n_steps colors of a single hue, light -> dark (low -> high substitution)."""
    anchor = np.array(to_rgb(anchor))
    ramp = LinearSegmentedColormap.from_list("", [anchor + (1 - anchor) * tint, anchor, anchor * (1 - shade)])
    return [ramp(t) for t in np.linspace(0, 1, n_steps)]

spectral = plt.get_cmap("Spectral")
parent_color = "#4d4d4d"
fig_width = 17.1
margin, gap, title_space = 0.05, 0.05, 0.3
isomer_markers = ["o", "s", "^"]
isomer_labels = ["1,2 · 1,2,3 · 1,2,3,4", "1,3 · 1,2,4 · 1,2,3,5", "1,4 · 1,3,5 · 1,2,4,5"]
max_ring_sub = 6
ring_gradients = {X: class_gradient(anchor, max_ring_sub) for X, anchor in {"N": spectral(0.92), "P": spectral(0.2)}.items()}
ring_colors = [parent_color] + [ring_gradients[X][n - 1] for X in heteroatoms for n in ring_counts[1:]]
ring_markers = ["*"] + [isomer_markers[i] for X in heteroatoms for i in ring_isomers[1:]]
bar_w, bar_gap = 0.2, 0.12
n_panels = len(model_paths)
side = 5.0  # cm, square; legend column is fig_width - legend_x = 1.9 cm
fig_h = round(margin + side + title_space + margin, 2)
to_fig = lambda x, y, w, h: [x / fig_width, y / fig_h, w / fig_width, h / fig_h]
fig = plt.figure(figsize=(fig_width * cm, fig_h * cm))
axes = [fig.add_axes(to_fig(margin + k * (side + gap), margin, side, side)) for k in range(n_panels)]
legend_x = margin + n_panels * (side + gap)
for ax, (name, pca) in zip(axes, pca_coordinates.items()):
    for i, (color, marker) in enumerate(zip(ring_colors, ring_markers)):
        pts = pca[mol_idx == i]
        is_parent = marker == "*"
        ax.scatter(pts[:, 0], pts[:, 1], color=color, marker=marker, s=12 * (3 if is_parent else 1),
                   linewidths=0.15, edgecolors="white", zorder=4 if is_parent else 3)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(MODEL_TITLES[name], fontsize=7, pad=2)
    for spine in ax.spines.values():
        spine.set_linewidth(0.25)

bar_bottom, bar_h = margin + 0.5 * side, 0.5 * side - title_space
for j, X in enumerate(heteroatoms):
    cmap = ListedColormap(ring_gradients[X])
    norm = BoundaryNorm(np.arange(0.5, len(ring_gradients[X]) + 1), cmap.N)
    cbar = fig.colorbar(matplotlib.cm.ScalarMappable(norm=norm, cmap=cmap),
                       cax=fig.add_axes(to_fig(legend_x + j * (bar_w + bar_gap), bar_bottom, bar_w, bar_h)),
                       orientation="vertical", ticks=range(1, max_ring_sub + 1))
    cbar.ax.set_title(X, fontsize=6, pad=3)
    cbar.ax.tick_params(labelsize=5, length=1.5, width=0.3, pad=1)
    cbar.ax.minorticks_off()
    cbar.outline.set_linewidth(0.3)
    if j < len(heteroatoms) - 1:
        cbar.ax.tick_params(right=False, labelright=False)
fig.legend(handles=[Line2D([0], [0], marker=m, linestyle="None", markersize=3, markerfacecolor="none", markeredgecolor=parent_color, markeredgewidth=0.5, label=label)
                    for m, label in zip(isomer_markers, isomer_labels)],
           title="X positions", title_fontsize=6, fontsize=5, frameon=False, loc="lower left",
           bbox_to_anchor=to_fig(legend_x, margin, 0, 0)[:2], handletextpad=0.3, labelspacing=0.5, borderaxespad=0, borderpad=0)
fig.savefig(OUTPUT_DIR / "substitution_ring_PCA.png", dpi=600)
plt.show()

## Cosine Similarity to Benzene

In [ ]:
display(pd.DataFrame(parent_similarities, index=ring_labels).rename(columns=MODEL_TITLES).round(3))